# Causal Inference: Estimating Effects (Optional Module)


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/M14-Causal-Inference-Estimation.ipynb)

**Optional module**

This module extends [Lecture 14](14-Causal-Inference.qmd). It is **not required**: it goes
deeper into material that Lecture 14 introduces.


[Lecture 14](14-Causal-Inference.qmd) showed how to choose an adjustment set $S$ with the
backdoor criterion, and named the estimators and natural-experiment designs that use it.
This module works those estimators on simulated data and looks at the designs in detail.


## Causal graphs: a recap

In [ ]:
# Setup for Colab and for a local clone of the Materials repo: make the course's
# helper modules importable, downloading any that are not already available.
import importlib.util, os, sys, urllib.request

_BASE = "https://raw.githubusercontent.com/tools4ds/DS701-Materials-FA26/main"
for _path in ["scripts/slideUtilities.py"]:
    _name = os.path.splitext(os.path.basename(_path))[0]
    if importlib.util.find_spec(_name) is not None:
        continue
    for _root in (".", ".."):                      # a clone: notebook in jupyter_notebooks/
        if os.path.exists(os.path.join(_root, _path)):
            sys.path.insert(0, os.path.abspath(os.path.dirname(os.path.join(_root, _path))))
            break
    else:                                          # Colab: fetch the file
        os.makedirs("modules", exist_ok=True)
        urllib.request.urlretrieve(f"{_BASE}/{_path}", os.path.join("modules", os.path.basename(_path)))
        if os.path.abspath("modules") not in sys.path:
            sys.path.insert(0, os.path.abspath("modules"))


In [ ]:
#| echo: false
#| fig-align: center
#| label: example-dag
import networkx as nx
import matplotlib.pyplot as plt
from slideUtilities import OKABE_ITO

def draw(ax, edges, pos, title="", node_color="#cde", node_size=1400, font_size=12):
    """Lecture-wide DAG style: pale nodes, dark outlines, black arrows."""
    G = nx.DiGraph(edges)
    colors = [node_color.get(v, "#cde") for v in G] if isinstance(node_color, dict) else node_color
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=node_size, font_size=font_size, arrowsize=22,
            edgecolors="#333", linewidths=1.2)
    ax.set_title(title, fontsize=11)

example_edges = [("A\nage", "X\nexercise"), ("A\nage", "Y\nheart"),
                 ("X\nexercise", "W\nweight"), ("W\nweight", "Y\nheart"),
                 ("M\nmotivation", "X\nexercise"), ("M\nmotivation", "D\ndiet"),
                 ("D\ndiet", "Y\nheart")]
example_pos = {"A\nage": (1, -1.3), "M\nmotivation": (-0.6, 1.4), "D\ndiet": (0.7, 1.0),
               "X\nexercise": (0, 0), "W\nweight": (1, 0), "Y\nheart": (2, 0)}

fig, ax = plt.subplots(figsize=(5.4, 3.6))
draw(ax, example_edges, example_pos, node_size=2600, font_size=9)
ax.set_xlim(-1.05, 2.4); ax.set_ylim(-1.75, 1.8)
plt.tight_layout(); plt.show()

From [Lecture 14](14-Causal-Inference.qmd):

* Every path is built from **chains**, **forks** and **colliders**. Chains and forks are
  open until you condition on them; colliders are closed until you do (or condition on a
  descendant).
* A **backdoor path** starts with an arrow *into* $X$. A set $S$ satisfies the **backdoor
  criterion** if it blocks every backdoor path and contains no descendant of $X$.
* Then **conditional ignorability** holds, $\{Y(0), Y(1)\} \perp\!\!\!\perp T \mid S$, and
  regression, stratification, matching or propensity scores can do the adjusting.


The running example: $X$ = exercise, $Y$ = heart health. When we turn to estimation, the
treatment is called $T$, as in Lecture 14's potential-outcomes notation.


## Worked example

Back to the exercise DAG ($X$ = exercise, $Y$ = heart health):

$$
A \to X,\; A \to Y,\; M \to X,\; M \to D,\; D \to Y,\; X \to W,\; W \to Y
$$

1. Paths from $X$ to $Y$: $X \to W \to Y$; $X \leftarrow A \to Y$; $X \leftarrow M \to D \to Y$.
2. Causal path: $X \to W \to Y$ — leave $W$ alone.
3. Backdoor paths: $X \leftarrow A \to Y$ (open, fork at $A$) and $X \leftarrow M \to D \to Y$
   (open, fork at $M$, chain at $D$). No colliders anywhere.
4. Block the first with $A$; block the second with $M$ *or* $D$.
5. Valid adjustment sets: $\{A, M\}$, $\{A, D\}$, or $\{A, M, D\}$. **Not** $\{A\}$ alone
   (leaves a backdoor open); **not** anything containing $W$ (a descendant of $X$).


In regression terms: `Y ~ X + A + M` recovers the total effect of exercise;
`Y ~ X + A + M + W` does not.


## Regression adjustment on a simulation

A confounder $Z$ (how sick a patient is) makes treatment more likely **and** makes the
outcome worse. The true effect of $T$ is $3$.

In [ ]:
#| echo: true
import numpy as np, pandas as pd, statsmodels.formula.api as smf

rng = np.random.default_rng(0)
n = 4000
Z = rng.normal(0, 1, n)                                         # sickness
T = (rng.uniform(size=n) < 1 / (1 + np.exp(-Z))).astype(int)   # sicker -> more likely treated
Y = 3.0 * T - 2.0 * Z + rng.normal(0, 1, n)                    # sicker -> worse; TRUE effect = 3.0
df = pd.DataFrame({"Y": Y, "T": T, "Z": Z})

naive = smf.ols("Y ~ T", data=df).fit().params["T"]
adj   = smf.ols("Y ~ T + Z", data=df).fit().params["T"]
print(f"True effect        = 3.00")
print(f"Naive (ignore Z)   = {naive:.2f}   <- biased")
print(f"Adjusted (+ Z)     = {adj:.2f}   <- recovers truth")

Predict the sign before you run it: $Z$ raises treatment *and* lowers $Y$, so the naive
estimate is biased **down**. The treatment looks less than half as effective as it is,
because the sickest were the ones treated.


## Stratification: Lecture 14's move on a continuous $Z$

Cut $Z$ into strata, compare treated and control *within* each, then average with the
population's stratum shares — Lecture 14's direct standardization, with $S$ in place of
kidney-stone size.

In [ ]:
#| echo: true
def stratified(df, k):
    """Direct standardization over k equal-size strata of Z."""
    q = pd.qcut(df["Z"], k, labels=False)
    means = df.groupby([q, "T"])["Y"].mean().unstack()     # rows: strata, cols: T = 0, 1
    within = means[1] - means[0]                           # effect inside each stratum
    return within, (within * q.value_counts(normalize=True).sort_index()).sum()

for k in (5, 10, 20):
    print(f"{k:2d} strata: standardized effect = {stratified(df, k)[1]:.2f}")

In [ ]:
#| echo: false
#| fig-align: center
#| label: strata
within5, std5 = stratified(df, 5)
std10, std20 = stratified(df, 10)[1], stratified(df, 20)[1]

fig, ax = plt.subplots(figsize=(6.4, 2.5))
labels = ["naive"] + [f"Q{i + 1}" for i in range(5)]
ax.bar(labels, [naive, *within5], color=["#999"] + [OKABE_ITO[5]] * 5)
for i, v in enumerate([naive, *within5]):
    ax.text(i, v - 0.1, f"{v:.2f}", ha="center", va="top", fontsize=9, color="white")
ax.axhline(3.0, ls="--", color="black", lw=1, label="true effect = 3.0")
ax.set_ylabel("treated − control")
ax.set_xlabel("quintile of Z (Q1 = healthiest)")
ax.set_ylim(0, 3.8)
ax.set_title(f"Standardized: 5 strata {std5:.2f}, 10 strata {std10:.2f}, "
             f"20 strata {std20:.2f}", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

Coarse strata leave **residual confounding** inside each stratum; finer strata close the
gap. Regression is the limit of infinitely fine strata — bought with a linearity assumption.


## Propensity scores in one slide

When $S$ is high-dimensional, matching directly is hard. Rosenbaum and Rubin showed it is
enough to match or weight on the scalar **propensity score** $e(S) = P(T{=}1\mid S)$
[@rosenbaum1983central].

In [ ]:
#| echo: true
from sklearn.linear_model import LogisticRegression

# Estimate propensity, then inverse-propensity weighting (IPW) for the ATE
ps = LogisticRegression().fit(df[["Z"]], df["T"]).predict_proba(df[["Z"]])[:, 1]
w = np.where(df["T"] == 1, 1 / ps, 1 / (1 - ps))
ate_ipw = (np.sum(w * df["T"] * df["Y"]) / np.sum(w * df["T"])
           - np.sum(w * (1 - df["T"]) * df["Y"]) / np.sum(w * (1 - df["T"])))
print(f"IPW estimate of ATE = {ate_ipw:.2f}   (true = 3.0)")

Inverse-propensity weighting builds a "pseudo-population" in which treatment is unrelated to
$S$ — mimicking a randomized experiment from observational data.


## Positivity: can we see it?

In [ ]:
#| echo: false
#| fig-align: center
#| label: overlap
fig, ax = plt.subplots(figsize=(6.4, 3.0))
bins = np.linspace(0, 1, 31)
ax.hist(ps[df["T"] == 1], bins=bins, alpha=0.5, color=OKABE_ITO[0], label="treated")
ax.hist(ps[df["T"] == 0], bins=bins, alpha=0.5, color=OKABE_ITO[2], label="control")
ax.set_xlabel("estimated propensity $e(Z)$"); ax.set_ylabel("patients")
ax.set_title("Overlap: both groups span nearly the same propensity range", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"treated: e from {ps[df['T'] == 1].min():.2f} to {ps[df['T'] == 1].max():.2f};  "
      f"control: {ps[df['T'] == 0].min():.2f} to {ps[df['T'] == 0].max():.2f};  "
      f"largest weight = {w.max():.1f}")

If one histogram had mass where the other has none, some units would have no comparable
counterpart. An $e$ of $0.01$ gives a weight of $100$: a tight standard error around a number
that means nothing.


## Same adjustment set, same answer

In [ ]:
#| echo: false
#| fig-align: center
#| label: estimators
fig, ax = plt.subplots(figsize=(6.4, 3.2))
names = ["naive\nY ~ T", "regression\nY ~ T + Z", "stratified\n20 strata of Z", "IPW on\ne(Z)"]
vals = [naive, adj, std20, ate_ipw]
ax.bar(names, vals, width=0.55, color=[OKABE_ITO[3]] + [OKABE_ITO[2]] * 3)
for i, v in enumerate(vals):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", va="bottom")
ax.axhline(3.0, ls="--", color="black", label="true effect = 3.0")
ax.set_ylabel("estimated effect of T"); ax.set_ylim(0, 3.8)
ax.set_title("Same data, same adjustment set — three estimators", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

Identification (which $S$) and estimation (which estimator) are separate decisions. A
*wrong* $S$ gives a wrong answer with a tight standard error — no estimator rescues a bad graph.


## When confounders are *unobserved*: natural experiments

Adjustment only works for confounders we can **measure**. When we can't, we look for
"nature's randomization" — sources of variation in treatment that are *as good as random*
[@angrist2014mastering].

Three workhorse designs:

* **Difference-in-Differences (DiD)**,
* **Instrumental Variables (IV)**,
* **Regression Discontinuity (RD)**.


## Difference-in-Differences

Compare the **change** in a treated group to the **change** in a control group over time.
This differences out any *fixed* differences between the groups.

In [ ]:
#| echo: false
#| fig-align: center
#| label: did
t = [-1, 0, 1]
treated_obs = [17, 20, 26]      # treated group: pre-period, before, after the policy
control     = [21, 24, 27]      # control group
counterfact = [20, 20 + (control[2] - control[1])]   # treated had it followed control's trend

fig, ax = plt.subplots(figsize=(5.0, 3.2))
ax.plot(t, treated_obs, "o-", label="Treated (observed)", color=OKABE_ITO[3])
ax.plot(t, control, "o-", label="Control", color=OKABE_ITO[2])
ax.plot([0, 1], counterfact, "o--", label="Treated (counterfactual)", color="#999")
ax.axvline(0.5, color="#ccc", lw=1)
ax.text(0.52, 16.6, "policy", color="#777", fontsize=8)
ax.annotate("", xy=(1, treated_obs[2]), xytext=(1, counterfact[1]),
            arrowprops=dict(arrowstyle="<->", color="black"))
ax.text(1.04, (treated_obs[2] + counterfact[1]) / 2, "DiD\n= 3", va="center")
ax.set_xticks(t); ax.set_xticklabels(["Pre", "Before", "After"])
ax.set_xlim(-1.3, 1.45)
ax.set_ylabel("Outcome"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

|  | Before | After | Change |
|:--|--:|--:|--:|
| Treated | 20 | 26 | +6 |
| Control | 24 | 27 | +3 |

DiD $= (26 - 20) - (27 - 24) = \mathbf{3}$.

Key assumption: **parallel trends** — absent treatment, both groups would have moved
together. The only check you can run is on the *pre* period: here both rose by 3.


Card and Krueger used DiD to show that a New Jersey minimum-wage increase did **not** reduce
fast-food employment relative to neighboring Pennsylvania [@card1994minimum].


## Instrumental variables & regression discontinuity

**Instrumental variable (IV)**

An **instrument** $L$ shifts the treatment and affects the outcome *only through* it:

$$
L \rightarrow T \rightarrow Y, \quad L \not\rightarrow Y \text{ directly}.
$$


- Vietnam **draft lottery** as an instrument for military service.
- **Distance to college** as an instrument for years of schooling.

$L$ acts like the coin flip we wish we had run.

Lecture 14's [instrument in graph terms](14-Causal-Inference.qmd#an-instrument-in-graph-terms)
states its three conditions as a DAG.

**Regression discontinuity (RD)**

Treatment switches on at a **cutoff** of a running variable. Units just above vs. just below
are comparable.

- Scholarship awarded at a **test-score threshold**.
- Class-size caps that trigger a new section.

Compare outcomes in a narrow window around the cutoff.


## Three designs, three assumptions

| Design | As-good-as-random variation | Assumption | How it fails |
|:--|:--|:--|:--|
| **DiD** | the *change* in treated vs. control | **parallel trends** | pre-period trends already diverge |
| **IV** | an instrument $L$ | **relevance** ($L \to T$), **exclusion** (no $L \to Y$), **independence** (no backdoor $L$–$Y$) | the lottery shifts earnings via draft *avoidance* (staying in college) |
| **RD** | a cutoff on a running variable | **nothing else jumps at the cutoff**; no precise manipulation | people bunch just above the threshold |

: {tbl-colwidths="[10,24,38,28]"}

Each design swaps "no unmeasured confounders" for a *different*, often more credible,
assumption that data cannot fully verify. No free lunch.


## Assumptions you can't skip

Every observational causal estimate relies on assumptions [@hernan2020whatif]:

* **Conditional ignorability** — $\{Y(0),Y(1)\} \perp\!\!\!\perp T \mid S$: the adjustment set
  blocks every backdoor path. The DAG argues it; data cannot check it.
* **Positivity (overlap)** — every unit has a nonzero chance of each treatment ($0 < e(S) < 1$).
* **SUTVA** — the assumption Lecture 14's switching equation
  $Y^{\text{obs}} = T\,Y(1) + (1-T)\,Y(0)$ quietly made: one unit's treatment doesn't affect
  another's outcome, and there is only one version of "treated."
* **Correct graph** — the DAG reflects reality.


Since ignorability is untestable, good practice includes a **sensitivity analysis**:
how strong would an unmeasured confounder have to be to overturn the conclusion?


## Tools and the modern workflow

* **[DoWhy](https://www.pywhy.org/dowhy/)** — model (DAG) → identify → estimate → refute; a
  clean four-step API [@dowhy2020].
* **[EconML](https://econml.azurewebsites.net/)** — ML-based heterogeneous treatment effects (CATE).
* **[dagitty](https://dagitty.net/)** — draw a DAG in the browser; it tells you the valid
  adjustment sets.
* Classical estimators live in **statsmodels**, **scikit-learn**, and **linearmodels**.


The workflow is always: **draw the graph → identify an estimand → estimate → challenge it
with robustness checks.** The graph comes *first*.


## Summary

1. **A causal DAG encodes your assumptions, not the data's.** Nodes are variables, arrows are
   direct causes, and every path is built from chains, forks, and colliders.
2. **Chains and forks are open until you condition on them; colliders are closed until you
   do** (or condition on a descendant). A path transmits association only if it is open.
3. **The backdoor procedure sorts every path into causal, backdoor or neither**, then blocks
   every open backdoor with non-descendants of $X$ — never a mediator, never a collider.
4. **Identification first, estimation second**: given a valid $S$, regression,
   stratification and IPW agree; with a wrong $S$ none of them can help.
5. **When confounders are unmeasured, use natural experiments** — DiD (parallel trends), IV
   (relevance, exclusion, independence), RD (nothing else jumps at the cutoff) — and state
   the standing assumptions (ignorability, positivity, SUTVA).


## Practice scenarios (self-check)

For each scenario, draw the DAG, name the **confounder**, the tempting-but-wrong control
(a mediator or collider), and a **design** you would use. Then open the intended answers.

**A. Product / tech.** Does adding a **"free shipping" badge** on product pages increase
**purchases**? You have historical logs; badges were shown more often on popular items.

**B. Health.** Does a new **fitness-tracking app** reduce users' **resting heart rate**? People
who download it are already more health-conscious.

**C. Education.** Does an **optional tutoring program** raise **final grades**? Students
self-select into tutoring, and struggling students are more likely to sign up.

**D. Policy.** Does a city's **new bike-lane network** reduce **traffic accidents**? It was built
in denser, wealthier neighborhoods first.

**Intended answers**

- **A.** Confounder: item **popularity** (drives both the badge and purchases). Tempting
  control: **"add-to-cart"** — a mediator on badge → add-to-cart → purchase. Design: an A/B
  test that randomizes the badge, or adjust for popularity with a propensity score.
- **B.** Confounder: **health-consciousness**. Tempting controls: **steps per day** (a
  mediator — the app works by getting people moving) and **"posted an app review"** (a
  collider — caused by both installing the app and by feeling the benefit). Design:
  randomize a free premium tier as an encouragement (an instrument), or DiD on users' heart
  rate before and after installing.
- **C.** This is Lecture 14's tutoring DAG. Confounder: **prior GPA** (struggling students
  sign up *and* score lower, so the naive estimate is biased down). Tempting controls:
  **study hours** (mediator) and **honor roll** (collider). Design: adjust for prior GPA; or
  RD if tutoring is assigned below a GPA cutoff.
- **D.** Confounder: neighborhood **density / wealth**. Tempting control: **cyclist volume** —
  a mediator (lanes → more cyclists → accidents). Design: DiD comparing accident trends in
  neighborhoods that got lanes early vs. late, checking pre-period trends.



## References

- [*Causal Inference: What If*](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) — Hernán & Robins (free PDF)
- [*Causal Inference: A Primer*](http://bayes.cs.ucla.edu/PRIMER/) — Pearl, Glymour & Jewell
- ["A Crash Course in Good and Bad Controls"](https://ftp.cs.ucla.edu/pub/stat_ser/r493.pdf) — Cinelli, Forney & Pearl
- [DoWhy documentation](https://www.pywhy.org/dowhy/) and [dagitty.net](https://dagitty.net/)

## Bibliography
